<a href="https://colab.research.google.com/github/Caleb-John1603/AI_5th_sem/blob/main/Week%207_Unification_1BM24CS095.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""
First-Order Logic unification with a step-by-step trace
=======================================================
Takes two atomic FOL statements (or terms), finds their Most General Unifier (MGU)
and shows every step of the algorithm (Martelli-Montanari style: delete /
decompose / orient / eliminate / conflict / occurs-check).

SYNTAX
  predicates, functions, constants : Capitalised   Knows(John, x)   Mother(y)   Marcus
  variables                        : lowercase     x  y  z1
  negation                         : leading ~     ~Loves(x, Mary)   (both sides must have the same sign)

USAGE
  python fol_unify.py "Knows(John, x)" "Knows(y, Mother(y))"
  python fol_unify.py                      # interactive
  python fol_unify.py --demo               # a set of examples
  --standardize   rename the variables of the 2nd statement apart first (x -> x_2)
                  e.g. Knows(John,x) vs Knows(x,Elizabeth) only unifies with this flag
"""
import argparse
import re
import sys


# ───────────────────────── parsing ─────────────────────────
# term: ("v", name)  |  ("f", name, (args...))      constant = f with no args
class ParseError(Exception):
    pass


TOKEN_RE = re.compile(r"\s*([A-Za-z0-9_']+|[(),~])")


def tokenize(text):
    text, pos, toks = text.strip(), 0, []
    while pos < len(text):
        m = TOKEN_RE.match(text, pos)
        if not m:
            raise ParseError(f"Unexpected character {text[pos]!r} in: {text}")
        toks.append(m.group(1))
        pos = m.end()
    return toks


class Parser:
    def __init__(self, text):
        self.text, self.toks, self.i = text, tokenize(text), 0

    def peek(self):
        return self.toks[self.i] if self.i < len(self.toks) else None

    def next(self):
        t = self.peek()
        if t is None:
            raise ParseError(f"Unexpected end of input: {self.text}")
        self.i += 1
        return t

    def parse(self):
        neg = False
        if self.peek() == "~":
            self.next()
            neg = True
        t = self.term()
        if self.peek() is not None:
            raise ParseError(f"Unexpected {self.peek()!r} in: {self.text}")
        return neg, t

    def term(self):
        name = self.next()
        if not re.fullmatch(r"[A-Za-z0-9_']+", name):
            raise ParseError(f"Bad symbol {name!r} in: {self.text}")
        if self.peek() == "(":
            self.next()
            args = [self.term()]
            while self.peek() == ",":
                self.next()
                args.append(self.term())
            if self.next() != ")":
                raise ParseError(f"Expected ')' in: {self.text}")
            return ("f", name, tuple(args))
        if name[0].islower():
            return ("v", name)
        return ("f", name, ())


# ───────────────────────── term utilities ─────────────────────────
def fmt(t):
    if t[0] == "v" or not t[2]:
        return t[1]
    return f"{t[1]}({', '.join(fmt(a) for a in t[2])})"


def variables(t):
    if t[0] == "v":
        return {t[1]}
    return set().union(*[variables(a) for a in t[2]]) if t[2] else set()


def subst(t, s):
    """Apply substitution dict {var-name: term} (single pass)."""
    if t[0] == "v":
        return s.get(t[1], t)
    return ("f", t[1], tuple(subst(a, s) for a in t[2]))


def rename_apart(t, suffix="_2"):
    if t[0] == "v":
        return ("v", t[1] + suffix)
    return ("f", t[1], tuple(rename_apart(a, suffix) for a in t[2]))


def fmt_theta(theta):
    if not theta:
        return "{}"
    return "{" + ", ".join(f"{v}/{fmt(t)}" for v, t in theta.items()) + "}"


def fmt_eqs(eqs):
    return "{ " + ",  ".join(f"{fmt(a)} = {fmt(b)}" for a, b in eqs) + " }" if eqs else "{ }  (empty)"


# ───────────────────────── unification with trace ─────────────────────────
def unify(t1, t2, quiet=False):
    """Returns (theta, None) on success or (None, reason) on failure; prints each step."""
    out = (lambda *a, **k: None) if quiet else print
    eqs = [(t1, t2)]
    theta = {}
    out(f"Start:  equations = {fmt_eqs(eqs)}")
    out(f"        θ = {{}}\n")
    step = 0

    while eqs:
        (a, b), rest = eqs[0], eqs[1:]
        step += 1
        out(f"Step {step}: take  {fmt(a)} = {fmt(b)}")

        # 1. DELETE: identical terms
        if a == b:
            out("        rule: DELETE  (both sides are identical)")
            eqs = rest

        # 2. ELIMINATE: variable on one side  (ORIENT first if the variable is on the right)
        elif a[0] == "v" or b[0] == "v":
            if a[0] != "v":
                out(f"        rule: ORIENT  (variable should be on the left)  ->  {fmt(b)} = {fmt(a)}")
                a, b = b, a
            if a[1] in variables(b):
                reason = f"occurs check fails: {a[1]} occurs inside {fmt(b)} (would need an infinite term)"
                out(f"        rule: OCCURS CHECK  ->  FAIL\n        {reason}")
                return None, reason
            binding = {a[1]: b}
            out(f"        rule: ELIMINATE  bind  {a[1]}/{fmt(b)}  and substitute it everywhere")
            theta = {k: subst(v, binding) for k, v in theta.items()}
            theta[a[1]] = b
            eqs = [(subst(x, binding), subst(y, binding)) for x, y in rest]

        # 3. function / predicate symbols
        elif a[1] != b[1]:
            reason = f"symbol clash: {a[1]} ≠ {b[1]}"
            out(f"        rule: CONFLICT  ->  FAIL\n        {reason}")
            return None, reason
        elif len(a[2]) != len(b[2]):
            reason = f"arity clash: {a[1]} takes {len(a[2])} argument(s) vs {len(b[2])}"
            out(f"        rule: CONFLICT  ->  FAIL\n        {reason}")
            return None, reason
        else:
            new = list(zip(a[2], b[2]))
            out(f"        rule: DECOMPOSE  same symbol {a[1]}/{len(a[2])}  ->  "
                + (", ".join(f"{fmt(x)} = {fmt(y)}" for x, y in new) if new else "(no arguments, nothing left)"))
            eqs = new + rest

        out(f"        equations = {fmt_eqs(eqs)}")
        out(f"        θ = {fmt_theta(theta)}\n")

    return theta, None


def run(s1, s2, standardize=False):
    neg1, t1 = Parser(s1).parse()
    neg2, t2 = Parser(s2).parse()
    print("=" * 72)
    print(f"Unify:  {'~' if neg1 else ''}{fmt(t1)}   with   {'~' if neg2 else ''}{fmt(t2)}")
    print("=" * 72)

    if standardize:
        t2 = rename_apart(t2)
        print(f"(standardised apart: 2nd statement is now {'~' if neg2 else ''}{fmt(t2)})\n")
    shared = variables(t1) & variables(t2)
    if shared and not standardize:
        print(f"note: both statements use the variable(s) {', '.join(sorted(shared))}; "
              f"they are treated as the SAME variable (use --standardize to separate them)\n")

    if neg1 != neg2:
        print("✘ FAIL: one statement is negated and the other is not.\n"
              "  (Complementary literals are RESOLVED, not unified - unify the positive parts instead.)\n")
        return None

    theta, reason = unify(t1, t2)
    if theta is None:
        print(f"✘ UNIFICATION FAILED - {reason}\n")
        return None

    r1, r2 = subst(t1, theta), subst(t2, theta)
    assert r1 == r2
    print("✔ UNIFIED")
    print(f"  MGU (most general unifier) θ = {fmt_theta(theta)}")
    print(f"  {'~' if neg1 else ''}{fmt(t1)}  θ  =  {'~' if neg1 else ''}{fmt(r1)}")
    print(f"  {'~' if neg2 else ''}{fmt(t2)}  θ  =  {'~' if neg2 else ''}{fmt(r2)}")
    print(f"  Unified statement:  {'~' if neg1 else ''}{fmt(r1)}\n")
    return theta


DEMOS = [
    ("Knows(John, x)", "Knows(John, Jane)", False),
    ("Knows(John, x)", "Knows(y, Mother(y))", False),
    ("Knows(John, x)", "Knows(x, Elizabeth)", False),          # fails: same variable x
    ("Knows(John, x)", "Knows(x, Elizabeth)", True),           # works once standardised apart
    ("P(x, F(y), B)", "P(A, F(G(z)), w)", False),
    ("Q(x, x)", "Q(y, F(y))", False),                          # occurs check
    ("Loves(John, x)", "Hates(John, x)", False),               # predicate clash
]


def main():
    ap = argparse.ArgumentParser(description="FOL unification with step-by-step trace")
    ap.add_argument("s1", nargs="?")
    ap.add_argument("s2", nargs="?")
    ap.add_argument("--standardize", action="store_true", help="rename 2nd statement's variables apart")
    ap.add_argument("--demo", action="store_true")
    a,unknown = ap.parse_known_args()
    try:
        if a.demo:
            for s1, s2, std in DEMOS:
                run(s1, s2, std)
        elif a.s1 and a.s2:
            run(a.s1, a.s2, a.standardize)
        else:
            print(__doc__.split("SYNTAX")[1].split("USAGE")[0])
            while True:
                try:
                    s1 = input("statement 1 (blank to quit)> ").strip()
                    if not s1:
                        break
                    s2 = input("statement 2> ").strip()
                    std = input("standardise apart? [y/N]> ").strip().lower() == "y"
                except EOFError:
                    break
                try:
                    run(s1, s2, std)
                except ParseError as e:
                    print(f"Parse error: {e}\n")
    except ParseError as e:
        sys.exit(f"Parse error: {e}")


if __name__ == "__main__":
    main()


  predicates, functions, constants : Capitalised   Knows(John, x)   Mother(y)   Marcus
  variables                        : lowercase     x  y  z1
  negation                         : leading ~     ~Loves(x, Mary)   (both sides must have the same sign)


statement 1 (blank to quit)> Knows(John,x)
statement 2> Knows(y,Mary)
standardise apart? [y/N]> y
Unify:  Knows(John, x)   with   Knows(y, Mary)
(standardised apart: 2nd statement is now Knows(y_2, Mary))

Start:  equations = { Knows(John, x) = Knows(y_2, Mary) }
        θ = {}

Step 1: take  Knows(John, x) = Knows(y_2, Mary)
        rule: DECOMPOSE  same symbol Knows/2  ->  John = y_2, x = Mary
        equations = { John = y_2,  x = Mary }
        θ = {}

Step 2: take  John = y_2
        rule: ORIENT  (variable should be on the left)  ->  y_2 = John
        rule: ELIMINATE  bind  y_2/John  and substitute it everywhere
        equations = { x = Mary }
        θ = {y_2/John}

Step 3: take  x = Mary
        rule: ELIMINATE  bind  x/Mary